# Energybae — Solar Load Calculator
### Electricity Bill to Excel Automation

**AI Intern Practical Task | Energybae, Pimpri, Pune**

---

This notebook automates the process of analysing a customer's electricity bill to calculate the correct solar system size.

**What it does:**
1. You upload or point to an electricity bill (PDF or image)
2. AI reads and extracts key fields — units consumed, load, tariff, bill amount, etc.
3. Solar system size, savings, and ROI are calculated
4. A formatted Excel report is generated and ready to download

**Supported bill types:** MSEDCL, BESCOM, TATA Power, CESC, and other Indian utilities.

---

## Step 0 — Install dependencies

Run this cell once to install all required packages.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai', 'pdfplumber', 'Pillow', 'openpyxl', 'ipywidgets'
])
print('All packages installed.')

## Step 1 — Configure your OpenAI API key

- **On Replit:** The `AI_INTEGRATIONS_OPENAI_API_KEY` is auto-configured — skip this cell.
- **Locally:** Set your `OPENAI_API_KEY` environment variable, or paste it below.

In [ ]:
import os

# ─── LOCAL ONLY: Paste your key here (remove before committing to Git!) ───
# os.environ['OPENAI_API_KEY'] = 'sk-...'  

# Check which key is available
replit_key = os.environ.get('AI_INTEGRATIONS_OPENAI_API_KEY')
openai_key = os.environ.get('OPENAI_API_KEY')

if replit_key:
    print('Using Replit AI Integrations key (no cost to you).')
elif openai_key:
    print('Using OPENAI_API_KEY from environment.')
else:
    print('WARNING: No OpenAI key found. Set OPENAI_API_KEY before proceeding.')

## Step 2 — Import the Solar Calculator modules

In [ ]:
import sys
import os

# Make sure the solar_calculator package is importable
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

from solar_calculator import extract_bill_data, calculate_solar_recommendation, generate_excel
print('Modules loaded successfully.')

## Step 3 — Interactive Bill Upload Widget

Run the cell below to get an interactive file uploader. Select a PDF or image of an electricity bill.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import io

# ── Upload widget ──────────────────────────────────────────────────────────
upload_widget = widgets.FileUpload(
    accept='.pdf,.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Bill',
    button_style='info',
    layout=widgets.Layout(width='300px')
)

process_btn = widgets.Button(
    description='Extract & Calculate',
    button_style='success',
    icon='bolt',
    layout=widgets.Layout(width='200px')
)

status_label = widgets.Label(value='Upload a bill and click Extract & Calculate.')
output_area = widgets.Output()

# ── State ──────────────────────────────────────────────────────────────────
state = {'bill_data': None, 'solar': None, 'excel_bytes': None}

def on_process_clicked(b):
    with output_area:
        clear_output()

        if not upload_widget.value:
            print('Please upload a file first.')
            return

        uploaded = list(upload_widget.value.values())[0]
        filename = uploaded['metadata']['name']
        file_bytes = bytes(uploaded['content'])

        status_label.value = f'Processing {filename}...'
        print(f'File: {filename} ({len(file_bytes)/1024:.1f} KB)')
        print('Step 1/3 — Extracting data with AI (10-20 seconds)...')

        try:
            bill_data = extract_bill_data(file_bytes, filename)
            state['bill_data'] = bill_data
        except Exception as e:
            print(f'ERROR during extraction: {e}')
            status_label.value = 'Extraction failed.'
            return

        print('Step 2/3 — Calculating solar recommendation...')
        solar = calculate_solar_recommendation(bill_data)
        state['solar'] = solar

        print('Step 3/3 — Generating Excel report...')
        try:
            excel_bytes = generate_excel(bill_data, solar)
            state['excel_bytes'] = excel_bytes
        except Exception as e:
            print(f'ERROR generating Excel: {e}')
            status_label.value = 'Excel generation failed.'
            return

        status_label.value = 'Done! See results in Step 4 below.'
        print('Done! Run Step 4 to see results and download the Excel file.')

process_btn.on_click(on_process_clicked)

display(
    widgets.HBox([upload_widget, process_btn]),
    status_label,
    output_area
)

## Step 4 — View extracted data and download the Excel report

Run after processing is complete.

In [ ]:
import base64
from IPython.display import display, HTML

bill_data = state.get('bill_data')
solar     = state.get('solar')
excel_bytes = state.get('excel_bytes')

if not bill_data:
    print('No data yet — run Step 3 first.')
else:
    # ── Customer Info ─────────────────────────────────────────────────────
    print('=' * 55)
    print('  CUSTOMER INFORMATION')
    print('=' * 55)
    print(f"  Consumer Name      : {bill_data['consumer_name']}")
    print(f"  Consumer Number    : {bill_data['consumer_number']}")
    print(f"  Meter Number       : {bill_data.get('meter_number') or 'N/A'}")
    print(f"  Distribution Co.   : {bill_data.get('distribution_company') or 'N/A'}")
    print(f"  Billing Month      : {bill_data['billing_month']}")
    print(f"  Tariff Category    : {bill_data['tariff_category']}")

    # ── Electricity Usage ─────────────────────────────────────────────────
    print()
    print('=' * 55)
    print('  ELECTRICITY USAGE')
    print('=' * 55)
    print(f"  Units Consumed     : {bill_data['units_consumed']:,.0f} kWh")
    print(f"  Sanctioned Load    : {bill_data['sanctioned_load']} kW")
    print(f"  Total Bill Amount  : Rs. {bill_data['total_bill_amount']:,.0f}")
    print(f"  Cost per Unit      : Rs. {solar['cost_per_unit']:.2f}/kWh")
    print(f"  Avg Daily Usage    : {solar['daily_units']:.2f} kWh/day")

    # ── Solar Recommendation ──────────────────────────────────────────────
    print()
    print('=' * 55)
    print('  SOLAR SYSTEM RECOMMENDATION')
    print('=' * 55)
    print(f"  System Size        : {solar['recommended_system_size_kw']} kWp")
    print(f"  Monthly Savings    : Rs. {solar['estimated_monthly_savings']:,}")
    print(f"  Annual Savings     : Rs. {solar['estimated_annual_savings']:,}")
    print(f"  System Cost (est.) : Rs. {solar['system_cost_inr']:,}")
    print(f"  Payback Period     : {solar['payback_period_years']} years")
    print(f"  CO2 Reduction      : {solar['co2_reduction_kg_per_year']:,} kg/year")
    print(f"  25-yr Net Benefit  : Rs. {solar['net_benefit_inr']:,}")

    # ── Download Link ─────────────────────────────────────────────────────
    if excel_bytes:
        b64 = base64.b64encode(excel_bytes).decode()
        fname = f"solar_load_{bill_data['consumer_number']}.xlsx"
        href = f'data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}'
        link = f'<a href="{href}" download="{fname}" style="background:#2C7A2C;color:white;padding:10px 20px;border-radius:6px;text-decoration:none;font-size:14px">Download Excel Report</a>'
        print()
        display(HTML(f'<div style="margin:16px 0">{link}</div>'))

---
## How it works — Technical walkthrough

### 1. File reading
- **PDF:** `pdfplumber` extracts text from each page. If the text is too short (scanned PDF), we raise an error and ask the user to upload an image instead.
- **Image (JPG/PNG):** The image is base64-encoded and sent directly to OpenAI's vision model.

### 2. AI extraction
- We send a structured prompt to **GPT-4o** (or GPT-5.4 on Replit) asking it to return a specific JSON object.
- The prompt lists every field we need and gives formatting rules (numbers must be numeric, date format, etc.).
- We use regex to extract the JSON block from the response.

### 3. Solar calculation
- **System size** = max(daily_units / 4.5 peak_sun_hours, sanctioned_load × 0.8), rounded up to 0.5 kWp.
- **Savings** = units covered by solar × cost per unit from the bill.
- **Payback** = system cost ÷ annual savings.
- **CO₂** = annual generation × 0.82 kg/kWh (India grid emission factor).

### 4. Excel generation
- `openpyxl` creates a formatted workbook with section headers, colour coding, and Indian currency formatting.
- All data is placed in input cells only — no formulas are overwritten.

---
## Script mode — Process a file from disk

If you prefer to run this without the widget, use the cell below directly. Change `BILL_FILE` to your file path.

In [ ]:
BILL_FILE = 'sample_bill.pdf'   # ← Change this to your file path
OUTPUT_FILE = 'solar_output.xlsx'

with open(BILL_FILE, 'rb') as f:
    file_bytes = f.read()

print(f'Loaded {BILL_FILE} ({len(file_bytes)/1024:.1f} KB)')

print('Extracting bill data...')
bill_data = extract_bill_data(file_bytes, BILL_FILE)

print('Calculating solar recommendation...')
solar = calculate_solar_recommendation(bill_data)

print('Generating Excel...')
excel_bytes = generate_excel(bill_data, solar)

with open(OUTPUT_FILE, 'wb') as f:
    f.write(excel_bytes)

print(f'Excel saved to {OUTPUT_FILE}')
print(f'Recommended system size: {solar["recommended_system_size_kw"]} kWp')
print(f'Estimated monthly savings: Rs. {solar["estimated_monthly_savings"]:,}')

---
*Energybae — Empowering People with Renewable Energy Solutions*  
*www.energybae.in | energybae.co@gmail.com | +91 9112233120*